# SpectraLite — Colab Bootstrap

Run this **once per new Colab runtime** (A100 / any GPU).

This notebook:
1. Clones or `git pull`s the full repo into `/content/Execution`
2. Installs Colab-safe deps (`requirements-colab.txt` — does **not** reinstall torch)
3. Verifies CUDA / GPU
4. Optionally runs the **Phase 0 smoke test in this same notebook** (recommended)

## IMPORTANT — how to open other `.ipynb` files

**Do NOT double-click** notebooks in the left Files panel. That opens a **text/raw editor**, not a runnable Colab notebook.

To open Phase0 / Phase1 / … in the **same runtime** without losing GPU/installs:

1. Keep this Bootstrap tab connected
2. `File` → `Open notebook` → **GitHub**
3. Repo `PrabinDevkota/Execution`, branch `main`
4. Click the phase notebook
5. If asked about runtime: **connect to the existing / hosted runtime** (do not create a brand-new machine unless needed)

Or stay here and run the optional Phase 0 section below.


### 0. Runtime check

Colab menu: **Runtime → Change runtime type → GPU (A100) → Save** before continuing.


In [ ]:
import sys
print("In Colab:", "google.colab" in sys.modules)

try:
    import torch
    print("Torch (before bootstrap):", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: No CUDA yet — select a GPU runtime, then delete runtime if needed.")
except ImportError:
    print("torch not imported yet (ok on a blank machine)")


### 1. Clone or pull repo

Re-run this cell whenever Cursor pushes new phases.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/PrabinDevkota/Execution.git"
CLONE_ROOT = Path("/content/Execution")
PACKAGE_ROOT = CLONE_ROOT / "SpectraLite"

if not PACKAGE_ROOT.is_dir():
    subprocess.check_call(["git", "clone", REPO_URL, str(CLONE_ROOT)])
else:
    subprocess.check_call(["git", "-C", str(CLONE_ROOT), "pull", "--ff-only"])

if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

%cd {PACKAGE_ROOT}
print("Repo ready:", PACKAGE_ROOT)
print("Has requirements-colab.txt:", (PACKAGE_ROOT / "requirements-colab.txt").is_file())
print("Has spectralite/:", (PACKAGE_ROOT / "spectralite").is_dir())
print("Notebooks:", sorted(p.name for p in (PACKAGE_ROOT / "notebooks").glob("*.ipynb")))


### 2. Install dependencies (Colab-safe)

Installs everything **except** `torch`, so Colab keeps its CUDA build.


In [ ]:
from spectralite.colab_setup import ensure_environment

# pull=False: already pulled above
repo_root = ensure_environment(pull=False, install=True, require_cuda_on_colab=True)
print("\nBootstrap complete.")


### 3. Quick import smoke test


In [ ]:
from spectralite import __version__, default_config
from spectralite.system import print_environment_report

print("spectralite", __version__)
print("default model", default_config().model_name)
print_environment_report()
print("\nOK — environment ready.")


### 4. Optional — Run Phase 0 here (same runtime, no second notebook)

Recommended if opening `Phase0_Setup.ipynb` keeps showing raw/text mode.
This runs the same Phase 0 checks using the `spectralite` package.


In [ ]:
from spectralite import default_config, set_seed, gpu
from spectralite.model_loader import load_model_and_tokenizer, print_model_load_summary, generate_text
from spectralite.model_analysis import print_full_model_analysis
from spectralite.utils import print_checklist, print_section, get_logger

cfg = default_config()
cfg.ensure_directories()
set_seed(cfg.seed)
logger = get_logger("spectralite.phase0")

mem_before = gpu.snapshot(label="Memory before loading")
gpu.print_memory_snapshot(mem_before)

model, tokenizer = load_model_and_tokenizer(config=cfg)
load_summary = print_model_load_summary(model, model_name=cfg.model_name)
mem_after_load = gpu.snapshot(label="Memory after loading")
gpu.print_memory_snapshot(mem_after_load)

analysis = print_full_model_analysis(model, model_name=cfg.model_name)
print("Linear layers:", analysis["num_linear_layers"])

generated = generate_text(
    model, tokenizer, cfg.smoke_prompt, max_new_tokens=cfg.max_new_tokens, do_sample=False
)
print_section("Test Inference")
print(generated)

mem_after_infer = gpu.snapshot(label="Memory after inference")
gpu.print_memory_timeline([mem_before, mem_after_load, mem_after_infer])

environment_ready = True
gpu_ready = gpu.is_cuda_available()
model_loaded = model is not None
inference_ok = bool(generated and generated.strip())

print_checklist(
    [
        ("Environment Ready", environment_ready),
        ("GPU Ready", gpu_ready),
        ("Model Loaded Successfully", model_loaded),
        ("Inference Successful", inference_ok),
        ("Phase 0 Complete", all([environment_ready, gpu_ready, model_loaded, inference_ok])),
    ]
)


### Next phases

For Phase 1+: prefer **File → Open notebook → GitHub** (never Files double-click).

Direct links (open in a new tab, then connect to this runtime if prompted):

- Bootstrap: `https://colab.research.google.com/github/PrabinDevkota/Execution/blob/main/SpectraLite/notebooks/Colab_Bootstrap.ipynb`
- Phase 0: `https://colab.research.google.com/github/PrabinDevkota/Execution/blob/main/SpectraLite/notebooks/Phase0_Setup.ipynb`
